<a href="https://colab.research.google.com/github/Anusha-thalla/Distress-detection-with-AI/blob/main/Distress%20detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import pandas as pd
import librosa

# Path to audio folder
audio_folder = "/content/Women_safety_dataset/raw_audio"

# Ensure the audio folder exists
os.makedirs(audio_folder, exist_ok=True)

# Store metadata
data = []

# Loop through files
for file_name in os.listdir(audio_folder):

    if file_name.endswith((".mp4", ".wav", ".mp3", ".m4a")):

        file_path = os.path.join(audio_folder, file_name)

        try:
            # Load audio
            audio, sr = librosa.load(file_path, sr=None)

            # Get duration
            duration = librosa.get_duration(y=audio, sr=sr)

            # Store details
            data.append({
                "file_name": file_name,
                "duration": round(duration, 2),
                "sample_rate": sr
            })

        except Exception as e:
            print(f"Error processing {file_name}: {e}")

# Create DataFrame
df = pd.DataFrame(data)

# Save metadata CSV
output_path = "women_safety_dataset/metadata/metadata.csv"
# Ensure the output directory exists
os.makedirs(os.path.dirname(output_path), exist_ok=True)
df.to_csv(output_path, index=False)

print("Metadata CSV created successfully!")

In [ ]:
from google.colab import files
import os

# Define upload path
upload_path = "/content/Women_safety_dataset/raw_audio"

# Upload files
uploaded = files.upload()

# Save uploaded files into raw_audio folder
for file_name in uploaded.keys():
    with open(os.path.join(upload_path, file_name), "wb") as f:
        f.write(uploaded[file_name])

print("Files uploaded successfully!")

In [ ]:
import os
from pydub import AudioSegment

# Input and output folders
input_folder = "/content/Women_safety_dataset/raw_audio"
output_folder = "/content/Women_safety_dataset/processed_audio"

# Create input and output folders if not exists
os.makedirs(input_folder, exist_ok=True)
os.makedirs(output_folder, exist_ok=True)

# Supported formats
supported_formats = (".mp4", ".mp3", ".m4a", ".wav")

# Process files
for file_name in os.listdir(input_folder):

    if file_name.endswith(supported_formats):

        input_path = os.path.join(input_folder, file_name)

        # Output filename
        output_name = os.path.splitext(file_name)[0] + ".wav"
        output_path = os.path.join(output_folder, output_name)

        try:
            # Load audio
            audio = AudioSegment.from_file(input_path)

            # Convert to mono
            audio = audio.set_channels(1)

            # Set sample rate to 16kHz
            audio = audio.set_frame_rate(16000)

            # Export as WAV
            audio.export(output_path, format="wav")

            print(f"Processed: {output_name}")

        except Exception as e:
            print(f"Error processing {file_name}: {e}")

print("Audio standardization completed!")

In [ ]:
os.listdir("/content/Women_safety_dataset/processed_audio")

In [ ]:
!apt-get install ffmpeg -y

In [ ]:
import os
from pydub import AudioSegment

# Paths
input_folder = "/content/Women_safety_dataset/raw_audio"
output_folder = "/content/Women_safety_dataset/processed_audio"

# Create output folder
os.makedirs(output_folder, exist_ok=True)

# Loop through files
for file_name in os.listdir(input_folder):

    if file_name.endswith(".mp4"):

        input_path = os.path.join(input_folder, file_name)

        # Output WAV name
        output_name = os.path.splitext(file_name)[0] + ".wav"
        output_path = os.path.join(output_folder, output_name)

        try:
            # Load MP4 audio
            audio = AudioSegment.from_file(input_path, format="mp4")

            # Convert to mono
            audio = audio.set_channels(1)

            # Set sample rate
            audio = audio.set_frame_rate(16000)

            # Export WAV
            audio.export(output_path, format="wav")

            print(f"Converted: {output_name}")

        except Exception as e:
            print(f"Error with {file_name}: {e}")

print("Conversion completed!")

In [ ]:
print(os.listdir("/content/Women_safety_dataset/processed_audio"))

In [ ]:
import os

processed_folder = "/content/Women_safety_dataset/processed_audio"

wav_files = [f for f in os.listdir(processed_folder) if f.endswith(".wav")]

print("Total WAV files:", len(wav_files))

In [ ]:
import os
import librosa
import numpy as np
import pandas as pd

# Path to processed WAV files
audio_folder = "/content/Women_safety_dataset/processed_audio"

# Store extracted features
features_list = []

# Loop through all WAV files
for file_name in os.listdir(audio_folder):

    if file_name.endswith(".wav"):

        file_path = os.path.join(audio_folder, file_name)

        try:
            # Load audio
            y, sr = librosa.load(file_path, sr=16000)

            # =========================
            # 1. MFCC Features
            # =========================
            mfccs = librosa.feature.mfcc(
                y=y,
                sr=sr,
                n_mfcc=13
            )

            mfccs_mean = np.mean(mfccs, axis=1)

            # =========================
            # 2. Pitch Extraction
            # =========================
            pitches, magnitudes = librosa.piptrack(
                y=y,
                sr=sr
            )

            pitch_values = pitches[magnitudes > np.median(magnitudes)]

            if len(pitch_values) > 0:
                pitch_mean = np.mean(pitch_values)
            else:
                pitch_mean = 0

            # =========================
            # 3. RMS Energy
            # =========================
            rms = librosa.feature.rms(y=y)
            rms_mean = np.mean(rms)

            # =========================
            # 4. Spectral Features
            # =========================

            # Spectral Centroid
            spectral_centroid = librosa.feature.spectral_centroid(
                y=y,
                sr=sr
            )
            spectral_centroid_mean = np.mean(spectral_centroid)

            # Spectral Bandwidth
            spectral_bandwidth = librosa.feature.spectral_bandwidth(
                y=y,
                sr=sr
            )
            spectral_bandwidth_mean = np.mean(spectral_bandwidth)

            # Zero Crossing Rate
            zcr = librosa.feature.zero_crossing_rate(y)
            zcr_mean = np.mean(zcr)

            # =========================
            # Store Features
            # =========================

            feature_dict = {
                "file_name": file_name,
                "pitch_mean": pitch_mean,
                "rms_energy": rms_mean,
                "spectral_centroid": spectral_centroid_mean,
                "spectral_bandwidth": spectral_bandwidth_mean,
                "zero_crossing_rate": zcr_mean
            }

            # Add MFCCs individually
            for i in range(13):
                feature_dict[f"mfcc_{i+1}"] = mfccs_mean[i]

            features_list.append(feature_dict)

            print(f"Processed: {file_name}")

        except Exception as e:
            print(f"Error processing {file_name}: {e}")

# Convert to DataFrame
features_df = pd.DataFrame(features_list)

# Save features CSV
features_df.to_csv("audio_features.csv", index=False)

print("\nFeature extraction completed!")

# Preview
print(features_df.head())

In [ ]:
!pip install transformers datasets torch scikit-learn pandas

In [ ]:
import pandas as pd

# Load dataset CSV file, specifying 'cp1252' encoding to handle potential character issues
df = pd.read_csv("/metadata.csv", encoding='cp1252')

# Display first 5 rows
print(df.head())

# Display dataset shape
print("Dataset Shape:", df.shape)

# Display column names
print("Columns:", df.columns)

# Check for missing values
print("\nMissing values:\n", df.isnull().sum())

In [ ]:
print(df.head(10))

In [ ]:
print(df["label"].unique())



In [ ]:
from sklearn.preprocessing import LabelEncoder

# Placeholder for labels, as 'label' column is missing in df
# You should replace 'unknown' with your actual labels based on your dataset
print(df["label"].unique())

# Encode labels to numerical format
le = LabelEncoder()
df["label_encoded"] = le.fit_transform(df["label"])

print("Labels encoded successfully!")
print(df.head())

In [ ]:
print(df["label"].value_counts())

In [ ]:
print(le.classes_)

In [ ]:
print(df[[
    "file_name",
    "Transcript ",
    "label",
    "label_encoded"
]].head(10))

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Encode labels
le = LabelEncoder()

df["label_encoded"] = le.fit_transform(df["label"])

# ==========================================
# SPLIT ENTIRE DATAFRAME
# ==========================================

train_df, test_df = train_test_split(

    df,

    test_size=0.2,

    random_state=42,

    stratify=df["label_encoded"]
)

# Reset index (VERY IMPORTANT)

train_df = train_df.reset_index(drop=True)

test_df = test_df.reset_index(drop=True)

print("Training Samples:", len(train_df))
print("Testing Samples:", len(test_df))

In [ ]:
# ==========================================
# CLEAN COLUMN NAMES
# ==========================================

# Remove hidden spaces and convert to lowercase

df.columns = df.columns.str.strip().str.lower()

train_df.columns = train_df.columns.str.strip().str.lower()

test_df.columns = test_df.columns.str.strip().str.lower()

features_df.columns = features_df.columns.str.strip().str.lower()

# ==========================================
# DEFINE AUDIO FEATURE COLUMNS
# ==========================================

audio_feature_columns = [

    'pitch_mean',

    'rms_energy',

    'spectral_centroid',

    'spectral_bandwidth',

    'zero_crossing_rate'

] + [f'mfcc_{i+1}' for i in range(13)]


In [ ]:
import pandas as pd
import re
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Define audio_feature_columns to ensure it's available in this scope
audio_feature_columns = ['pitch_mean', 'rms_energy', 'spectral_centroid', 'spectral_bandwidth', 'zero_crossing_rate',
                         'mfcc_1', 'mfcc_2', 'mfcc_3', 'mfcc_4', 'mfcc_5', 'mfcc_6', 'mfcc_7', 'mfcc_8', 'mfcc_9',
                         'mfcc_10', 'mfcc_11', 'mfcc_12', 'mfcc_13']

# Assuming 'df' and 'features_df' are available from global scope from previous cells.
# Make a copy of df to avoid modifying the original global df directly in case it's used elsewhere later.
temp_df = df.copy()

# Standardize filenames in temp_df to match features_df (e.g., .mp4 to .wav)
# This step is usually done in cell 25794121, but repeated here for self-containment.
temp_df['file_name'] = temp_df['file_name'].apply(
    lambda x: re.sub(r'\.mp4$', '.wav', x)
)

# Sort dataframes before merge for consistency
# This step is usually done in cell 25794121, but repeated here for self-containment.
temp_df = temp_df.sort_values("file_name").reset_index(drop=True)
features_df_sorted = features_df.sort_values("file_name").reset_index(drop=True)

# Merge audio features into the main dataframe
# This step is usually done in cell 25794121, but repeated here for self-containment.
merged_df = pd.merge(
    temp_df,
    features_df_sorted,
    on='file_name',
    how='left'
)
merged_df = merged_df.reset_index(drop=True)

# Remove hidden spaces and convert to lowercase for all columns in the merged_df
# This step is usually done in cell BPNfMn_bodRu, but repeated here for self-containment.
merged_df.columns = merged_df.columns.str.strip().str.lower()

# Split the merged dataframe into training and testing sets
# This step is usually done in cell Zus5A2LJ0cix, but repeated here for self-containment
# The merged_df now contains all the audio features and the label_encoded column.
train_df, test_df = train_test_split(
    merged_df,
    test_size=0.2,
    random_state=42,
    stratify=merged_df["label_encoded"]
)

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

# Text data
train_texts = train_df["transcript"].tolist()
test_texts = test_df["transcript"].tolist()

# Audio features
X_audio_train = train_df[audio_feature_columns].values
X_audio_test = test_df[audio_feature_columns].values

# Labels
y_train = train_df["label_encoded"].values
y_test = test_df["label_encoded"].values

print("Train/Test preparation completed!")

In [ ]:
features_df = features_df.sort_values("file_name").reset_index(drop=True)

In [ ]:
import re

# Standardize filenames
df['file_name'] = df['file_name'].apply(

    lambda x: re.sub(r'\.mp4$', '.wav', x)
)

# Sort both dataframes BEFORE merge

df = df.sort_values("file_name").reset_index(drop=True)

features_df = features_df.sort_values("file_name").reset_index(drop=True)

# Merge audio features

df = pd.merge(

    df,
    features_df,

    on='file_name',

    how='left'
)

# Reset index AFTER merge

df = df.reset_index(drop=True)

print("Merged DataFrame:")
display(df.head())

In [ ]:
df = df.sort_values("file_name").reset_index(drop=True)

In [ ]:
from transformers import DistilBertTokenizer

# Load pretrained DistilBERT tokenizer
tokenizer = DistilBertTokenizer.from_pretrained(
    "distilbert-base-uncased"
)

print("Tokenizer loaded successfully!")
train_texts = train_df["transcript"]

test_texts = test_df["transcript"]

# Tokenize training texts

train_encodings = tokenizer(

    train_texts.tolist(),

    truncation=True,

    padding=True,

    max_length=64
)

# Tokenize testing texts

test_encodings = tokenizer(

    test_texts.tolist(),

    truncation=True,

    padding=True,

    max_length=64
)

print("Text tokenization completed!")

In [ ]:
from transformers import DistilBertModel
import torch

# Load pretrained DistilBERT base model
bert_model = DistilBertModel.from_pretrained(
    "distilbert-base-uncased"
)

bert_model.eval()

In [ ]:
X_audio_train = train_df[audio_feature_columns].values

X_audio_test = test_df[audio_feature_columns].values

In [ ]:
train_texts = train_df["transcript"].tolist()

test_texts = test_df["transcript"].tolist()

In [ ]:
train_encodings = tokenizer(

    train_texts,

    truncation=True,

    padding=True,

    max_length=64,

    return_tensors="pt"
)

test_encodings = tokenizer(

    test_texts,

    truncation=True,

    padding=True,

    max_length=64,

    return_tensors="pt"
)

In [ ]:
with torch.no_grad():

    bert_model.eval()

    train_output = bert_model(

        input_ids=train_encodings['input_ids'],

        attention_mask=train_encodings['attention_mask']
    )

train_embeddings = train_output.last_hidden_state[:, 0, :]

In [ ]:
with torch.no_grad():

    test_output = bert_model(

        input_ids=test_encodings['input_ids'],

        attention_mask=test_encodings['attention_mask']
    )

test_embeddings = test_output.last_hidden_state[:, 0, :]

In [ ]:
X_audio_train = torch.tensor(
    X_audio_train,
    dtype=torch.float32
)

X_audio_test = torch.tensor(
    X_audio_test,
    dtype=torch.float32
)

In [ ]:
y_train = torch.tensor(
    train_df["label_encoded"].values,
    dtype=torch.long
)

y_test = torch.tensor(
    test_df["label_encoded"].values,
    dtype=torch.long
)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class CHGAFNet(nn.Module):

    def __init__(self, audio_dim, text_dim, hidden_dim, num_classes):

        super(CHGAFNet, self).__init__()

        # ==========================================
        # AUDIO ATTENTION
        # ==========================================

        self.audio_attention = nn.Sequential(

            nn.Linear(audio_dim, hidden_dim),

            nn.Tanh(),

            nn.Linear(hidden_dim, audio_dim),

            nn.Softmax(dim=1)
        )

        # ==========================================
        # TEXT ATTENTION
        # ==========================================

        self.text_attention = nn.Sequential(

            nn.Linear(text_dim, hidden_dim),

            nn.Tanh(),

            nn.Linear(hidden_dim, text_dim),

            nn.Softmax(dim=1)
        )

        # ==========================================
        # CONFIDENCE GATES
        # ==========================================

        self.audio_gate = nn.Sequential(

            nn.Linear(audio_dim, 1),

            nn.Sigmoid()
        )

        self.text_gate = nn.Sequential(

            nn.Linear(text_dim, 1),

            nn.Sigmoid()
        )

        # ==========================================
        # FUSION LAYER
        # ==========================================

        self.fusion_layer = nn.Linear(

            audio_dim + text_dim,

            hidden_dim
        )

        # Layer normalization for stability

        self.layer_norm = nn.LayerNorm(hidden_dim)

        # ==========================================
        # CLASSIFIER
        # ==========================================

        self.classifier = nn.Sequential(

            nn.ReLU(),

            nn.Dropout(0.5),

            nn.Linear(hidden_dim, hidden_dim // 2),

            nn.ReLU(),

            nn.Dropout(0.3),

            nn.Linear(hidden_dim // 2, num_classes)
        )

    def forward(self, audio_x, text_x):

        # ==========================================
        # AUDIO ATTENTION
        # ==========================================

        audio_attn_weights = self.audio_attention(audio_x)

        audio_attended = audio_x * audio_attn_weights

        # ==========================================
        # TEXT ATTENTION
        # ==========================================

        text_attn_weights = self.text_attention(text_x)

        text_attended = text_x * text_attn_weights

        # ==========================================
        # CONFIDENCE GATES
        # ==========================================

        audio_conf = self.audio_gate(audio_attended)

        text_conf = self.text_gate(text_attended)

        # ==========================================
        # NORMALIZED CONFIDENCE
        # ==========================================

        total_conf = audio_conf + text_conf

        epsilon = 1e-8

        audio_weight = audio_conf / (total_conf + epsilon)

        text_weight = text_conf / (total_conf + epsilon)

        # ==========================================
        # ADAPTIVE GATED FUSION
        # ==========================================

        weighted_audio = audio_attended * audio_weight

        weighted_text = text_attended * text_weight

        fused = torch.cat(

            [weighted_audio, weighted_text],

            dim=1
        )

        # ==========================================
        # FUSION TRANSFORMATION
        # ==========================================

        fused = self.fusion_layer(fused)

        fused = self.layer_norm(fused)

        # ==========================================
        # FINAL CLASSIFICATION
        # ==========================================

        output = self.classifier(fused)

        return output

In [ ]:
model = CHGAFNet(

    audio_dim = X_audio_train.shape[1],

    text_dim = train_embeddings.shape[1],

    hidden_dim = 32,

    num_classes = 3
)

print(model)

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
import torch.nn as nn

# ==========================================
# COMPUTE CLASS WEIGHTS
# ==========================================

class_weights = compute_class_weight(

    class_weight='balanced',

    classes=np.unique(train_df["label_encoded"]),

    y=train_df["label_encoded"]
)

# Convert to tensor

class_weights = torch.tensor(

    class_weights,

    dtype=torch.float32
)

print("Class Weights:", class_weights)

# ==========================================
# LOSS FUNCTION
# ==========================================

criterion = nn.CrossEntropyLoss(

    weight=class_weights
)

# ==========================================
# OPTIMIZER
# ==========================================

optimizer = torch.optim.Adam(

    model.parameters(),

    lr=0.001,

    weight_decay=1e-4
)

In [ ]:
print(X_audio_test.shape)

print(test_embeddings.shape)

print(y_test.shape)

In [ ]:
display(df.head())

In [ ]:
print(df["file_name"].duplicated().sum())

In [ ]:
from torch.utils.data import TensorDataset

train_dataset = TensorDataset(

    X_audio_train,

    train_embeddings,

    y_train
)

print("Train dataset created!")

In [ ]:
test_dataset = TensorDataset(

    X_audio_test,

    test_embeddings,

    y_test
)

print("Test dataset created!")
from torch.utils.data import DataLoader

train_loader = DataLoader(

    train_dataset,

    batch_size=4,

    shuffle=True
)

test_loader = DataLoader(

    test_dataset,

    batch_size=4,

    shuffle=False
)

print("DataLoaders created!")

In [ ]:
for audio_batch, text_batch, labels_batch in train_loader:

    print("Audio Shape:", audio_batch.shape)

    print("Text Shape:", text_batch.shape)

    print("Labels Shape:", labels_batch.shape)

    break

In [ ]:
num_epochs = 10

train_losses = []

for epoch in range(num_epochs):

    model.train()

    total_loss = 0

    for audio_batch, text_batch, labels_batch in train_loader:

        # ==========================================
        # FORWARD PASS
        # ==========================================

        outputs = model(

            audio_batch,

            text_batch
        )

        # ==========================================
        # LOSS
        # ==========================================

        loss = criterion(

            outputs,

            labels_batch
        )

        # ==========================================
        # BACKPROPAGATION
        # ==========================================

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    train_losses.append(avg_loss)

    print(f"Epoch [{epoch+1}/{num_epochs}] Loss: {avg_loss:.4f}")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6,4))

plt.plot(train_losses)

plt.xlabel("Epoch")

plt.ylabel("Loss")

plt.title("CHGAF-Net Training Loss")

plt.show()

In [ ]:
model.eval()

print("Model set to evaluation mode!")

In [ ]:
from sklearn.metrics import accuracy_score
import numpy as np
import torch

# Store predictions

all_predictions = []

all_actuals = []

# Disable gradient computation

with torch.no_grad():

    for audio_batch, text_batch, labels_batch in test_loader:

        # Forward pass

        outputs = model(

            audio_batch,

            text_batch
        )

        # Predicted class index

        _, predicted = torch.max(outputs, 1)

        # Store predictions

        all_predictions.extend(predicted.cpu().numpy())

        all_actuals.extend(labels_batch.cpu().numpy())

print("Prediction completed!")

In [ ]:
# Decode labels back to text

predicted_labels = le.inverse_transform(all_predictions)

actual_labels = le.inverse_transform(all_actuals)

# Display predictions

for actual, pred in zip(actual_labels, predicted_labels):

    print(f"Actual: {actual} | Predicted: {pred}")

In [ ]:
from sklearn.metrics import classification_report

print(

    classification_report(

        actual_labels,

        predicted_labels
    )
)

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import numpy as np

# ==========================================
# COMPUTE CONFUSION MATRIX
# ==========================================

cm = confusion_matrix(

    actual_labels,

    predicted_labels,

    labels=le.classes_
)

# ==========================================
# PLOT PROFESSIONAL CONFUSION MATRIX
# ==========================================

fig, ax = plt.subplots(figsize=(7,6))

im = ax.imshow(cm)

# Labels

classes = le.classes_

ax.set_xticks(np.arange(len(classes)))

ax.set_yticks(np.arange(len(classes)))

ax.set_xticklabels(classes, fontsize=12)

ax.set_yticklabels(classes, fontsize=12)

# Axis Labels

ax.set_xlabel("Predicted Label", fontsize=14, fontweight='bold')

ax.set_ylabel("True Label", fontsize=14, fontweight='bold')

# Title

ax.set_title(

    "CHGAF-Net Confusion Matrix",

    fontsize=16,

    fontweight='bold'
)

# Add values inside cells

for i in range(len(classes)):

    for j in range(len(classes)):

        ax.text(

            j,

            i,

            cm[i, j],

            ha="center",

            va="center",

            fontsize=14,

            fontweight='bold',

            color="white" if cm[i, j] > cm.max()/2 else "black"
        )

# Add color bar

cbar = fig.colorbar(im)

cbar.ax.tick_params(labelsize=12)

# Tight layout

plt.tight_layout()

# Save high-resolution image

plt.savefig(

    "CHGAF_confusion_matrix.png",

    dpi=300,

    bbox_inches='tight'
)

plt.show()

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

import pandas as pd

# ==========================================
# CALCULATE METRICS
# ==========================================

accuracy = accuracy_score(

    actual_labels,

    predicted_labels
)

precision = precision_score(

    actual_labels,

    predicted_labels,

    average='weighted'
)

recall = recall_score(

    actual_labels,

    predicted_labels,

    average='weighted'
)

f1 = f1_score(

    actual_labels,

    predicted_labels,

    average='weighted'
)

# ==========================================
# CONVERT TO PERCENTAGE
# ==========================================

accuracy_pct = accuracy * 100

precision_pct = precision * 100

recall_pct = recall * 100

f1_pct = f1 * 100

# ==========================================
# CREATE RESULTS TABLE
# ==========================================

results_df = pd.DataFrame({

    "Metric": [

        "Accuracy",

        "Precision",

        "Recall",

        "F1-Score"
    ],

    "Value (%)": [

        round(accuracy_pct, 2),

        round(precision_pct, 2),

        round(recall_pct, 2),

        round(f1_pct, 2)
    ]
})

# ==========================================
# DISPLAY RESULTS
# ==========================================

print("\nCHGAF-Net Performance Evaluation\n")

display(results_df)

In [ ]:
import os

print(os.listdir())

In [ ]:
import nbformat as nbf
import os

# NOTE: Replace 'Distress detection.ipynb' with the actual name of the notebook you want to clean.
# Ensure the notebook is in the same directory or provide its full path.
notebook_to_clean = "Distress detection.ipynb" # Placeholder - replace with your actual notebook name

# Check if the notebook exists before trying to read it
if not os.path.exists(notebook_to_clean):
    print(f"Error: Notebook '{notebook_to_clean}' not found. Please ensure the file exists and the path is correct.")
else:
    # Load notebook
    nb = nbf.read(notebook_to_clean, as_version=4)

    # Remove widget metadata completely
    if "widgets" in nb["metadata"]:
        del nb["metadata"]["widgets"]

    # Clear outputs from all cells
    for cell in nb.cells:
        if cell.cell_type == "code":
            cell.outputs = []
            cell.execution_count = None

    # Save cleaned notebook to a new file to avoid overwriting the original
    output_notebook_name = f"cleaned_{notebook_to_clean}"
    nbf.write(nb, output_notebook_name)

    print(f"Cleaned notebook '{output_notebook_name}' saved!")